<a href="https://colab.research.google.com/github/Sakhiur2022/Signal/blob/main/notebooks/02_data_cleaning/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Data Cleaning

### Decisions applied
- Binary encoding for gender, disability, digital_use, social_participation, agri_employment
- Ordinal encoding for disability_severity
- Keep disability_type as binary indicator flags (preserves story)
- Create `digital_status` (0 = no digital, 1 = digital but no internet, 2 = uses internet)
- Coarsen province → region (Java, Sumatra, Kalimantan, Sulawesi, Bali_Nusa, Eastern)
- Create income_class (tertiles)
- Drop derived / redundant columns
- Prefer weekly_workhours over monthly_workhours

## 1. Import Libraries


In [3]:
import pandas as pd
import numpy as np

# 2. Load Dataset

In [1]:
! wget https://huggingface.co/datasets/Sakhiur/signal/resolve/main/data_in_brief.csv

--2026-08-25 08:20:31--  https://huggingface.co/datasets/Sakhiur/signal/resolve/main/data_in_brief.csv
Resolving huggingface.co (huggingface.co)... 13.226.251.66, 13.226.251.81, 13.226.251.20, ...
Connecting to huggingface.co (huggingface.co)|13.226.251.66|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6a7c7183f2c81824bdfb6fa9/cdf287498ff2282521a8063be5394663f811af2c15b316b677afd23a9038423d?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27data_in_brief.csv%3B+filename%3D%22data_in_brief.csv%22%3B&response-content-type=text%2Fcsv&X-Xet-Cas-Uid=public&user_id=public&Expires=1787649631&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmE3YzcxODNmMmM4MTgyNGJkZmI2ZmE5L2NkZjI4NzQ5OGZmMjI4MjUyMWE4MDYzYmU1Mzk0NjYzZjgxMWFmMmMxNWIzMTZiNjc3YWZkMjNhOTAzODQyM2RcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmVzcG9uc2UtY29udGVudC10eXBlPSomWC1YZXQtQ2FzLVVpZD1wdWJsaWMmdXNlcl9

In [4]:
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

In [5]:
df = pd.read_csv("data_in_brief.csv")

print(df.columns.tolist())
df.head()

['age', 'gender', 'education_years', 'disability', 'disability_severity', 'disability_type', 'monthly_income', 'weekly_workhours', 'agri_productivity_trimmed', 'digital_use', 'social_participation', 'agri_employment', 'province', 'log_monthly_income', 'monthly_workhours', 'internet_use', 'digital_index', 'social_participation_index', 'agri_productivity']


,age,gender,education_years,disability,disability_severity,disability_type,monthly_income,weekly_workhours,agri_productivity_trimmed,digital_use,social_participation,agri_employment,province,log_monthly_income,monthly_workhours,internet_use,digital_index,social_participation_index,agri_productivity
0,78.0,Male,6.0,Non-disabled,Non-disabled,NaN,NaN,NaN,NaN,Non-digital user,Participates socially,Non-agricultural worker,11.0,NaN,NaN,NaN,0.0,0.333333,NaN
1,65.0,Female,6.0,Non-disabled,Non-disabled,NaN,NaN,NaN,NaN,Non-digital user,Participates socially,Non-agricultural worker,11.0,NaN,NaN,NaN,0.0,0.333333,NaN
2,82.0,Male,6.0,Non-disabled,Non-disabled,NaN,NaN,NaN,NaN,Non-digital user,No participation,Non-agricultural worker,11.0,NaN,NaN,NaN,0.0,0.000000,NaN
3,65.0,Female,6.0,Non-disabled,Non-disabled,NaN,NaN,NaN,NaN,Non-digital user,No participation,Non-agricultural worker,11.0,NaN,NaN,NaN,0.0,0.000000,NaN
4,31.0,Male,12.0,Non-disabled,Non-disabled,NaN,780000.0,35.0,22285.715,Digital user,Participates socially,Non-agricultural worker,11.0,13.567049,140.0,Use internet,1.0,0.333333,22285.715


## 2. Quick Inspection

In [6]:
print("Missing values:\n")
print(df.isnull().sum().sort_values(ascending=False))

print("\n\nDtypes:\n")
print(df.dtypes)

print("\n\nUnique values (categorical candidates):")
for col in ["gender", "disability", "disability_severity", "disability_type",
            "digital_use", "internet_use", "social_participation",
            "agri_employment", "digital_index", "social_participation_index"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False))

Missing values:

disability_type               692970
internet_use                  502103
agri_productivity_trimmed     329504
log_monthly_income            327963
monthly_income                325770
agri_productivity             325770
monthly_workhours             249197
weekly_workhours              249197
education_years                    0
age                                0
gender                             0
social_participation               0
digital_use                        0
disability_severity                0
disability                         0
agri_employment                    0
province                           0
digital_index                      0
social_participation_index         0
dtype: int64


Dtypes:

age                           float64
gender                         object
education_years               float64
disability                     object
disability_severity            object
disability_type                object
monthly_income              

## 3. Cleaning Pipeline

In [7]:
df_clean = df.copy()
print("Starting shape:", df_clean.shape)

Starting shape: (751622, 19)


In [8]:
df_clean[df_clean["monthly_income"].isna()]["weekly_workhours"].describe()
df_clean[df_clean["monthly_income"].isna()]["weekly_workhours"].isna().sum()

np.int64(249197)

### 3.1 Binary Encoding

In [9]:
# Gender
df_clean["gender_male"] = (df_clean["gender"].str.lower().str.strip() == "male").astype(int)

# Disability (binary)
df_clean["disabled"] = (df_clean["disability"].str.lower().str.strip() == "disabled").astype(int)

#Digital use (temporary, will be replaced by digital_status)
df_clean["is_digital_user"] = (
    df_clean["digital_use"].str.lower().str.strip() == "digital user"
).astype(int)

#Social participation
df_clean["social_participant"] = (
    df_clean["social_participation"].str.lower().str.contains("participates socially", na=False)
).astype(int)

#Agricultural employment
df_clean["agri_worker"] = (
    df_clean["agri_employment"].str.lower().str.strip() == "agricultural worker"
).astype(int)

print("Binary columns created: gender_male, disabled, is_digital_user, social_participant, agri_worker")
print("Gender:")
print(df_clean[["gender", "gender_male"]].value_counts())
print("\nDisability:")
print(df_clean[["disability", "disabled"]].value_counts())
print("\nDigital use:")
print(df_clean[["digital_use", "is_digital_user"]].value_counts())
print("\nSocial participation:")
print(df_clean[["social_participation", "social_participant"]].value_counts())
print("\nAgricultural employment:")
print(df_clean[["agri_employment", "agri_worker"]].value_counts())

Binary columns created: gender_male, disabled, is_digital_user, social_participant, agri_worker
Gender:
gender  gender_male
Female  0              379145
Male    1              372477
Name: count, dtype: int64

Disability:
disability    disabled
Non-disabled  0           692970
Disabled      1            58652
Name: count, dtype: int64

Digital use:
digital_use       is_digital_user
Non-digital user  0                  502103
Digital user      1                  249519
Name: count, dtype: int64

Social participation:
social_participation   social_participant
Participates socially  1                     624066
No participation       0                     127556
Name: count, dtype: int64

Agricultural employment:
agri_employment          agri_worker
Non-agricultural worker  0              735636
Agricultural worker      1               15986
Name: count, dtype: int64


### 3.2 Disability Severity (Ordinal)

In [ ]:
severity_map = {
    "non-disabled": 0,
    "non disabled": 0,
    "mild disability": 1,
    "mild": 1,
    "severe disability": 2,
    "severe": 2,
    "total disability": 3,
    "total": 3,
}

df_clean["disability_severity_ord"] = (
    df_clean["disability_severity"]
    .str.lower()
    .str.strip()
    .map(severity_map)
)

print("disability_severity_ord value counts:")
print(df_clean["disability_severity_ord"].value_counts(dropna=False))

disability_severity_ord value counts:
disability_severity_ord
0    692970
1     44711
2      9924
3      4017
Name: count, dtype: int64


### 3.3 Disability Type → Binary Flags (keep the story)

In [10]:
# Inspect actual values first
print("Unique disability_type values:")
print(df_clean["disability_type"].value_counts(dropna=False))

Unique disability_type values:
disability_type
NaN                         692970
Vision difficulty            18981
Mental/other difficulty      13080
Walking difficulty           12749
Hearing difficulty            6130
Hand/finger difficulty        4714
Communication difficulty      2998
Name: count, dtype: int64


In [11]:
# First, check the real values in your data
print("Unique disability_type values:")
print(df_clean["disability_type"].value_counts(dropna=False))
print()

# Mapping: actual value in data → new clean column name
type_mapping = {
    "vision difficulty": "disability_vision",
    "mental/other difficulty": "disability_mental",
    "walking difficulty": "disability_walking",
    "hearing difficulty": "disability_hearing",
    "communication difficulty": "disability_communication",
    "hand/finger difficulty": "disability_hand_finger"
}

# Initialise all flags to 0
for new_col in type_mapping.values():
    df_clean[new_col] = 0

# Set flag = 1 where the type matches
for raw_value, new_col in type_mapping.items():
    mask = (
        df_clean["disability_type"]
        .str.lower()
        .str.strip() == raw_value
    )
    df_clean.loc[mask, new_col] = 1

# Create a flag for NaN disability_type, renamed to 'no_disability'
df_clean["no_disability"] = df_clean["disability_type"].isna().astype(int)

# Check results
flag_cols = list(type_mapping.values()) + ["no_disability"]

print("Disability type flags created:")
print(flag_cols)
print("\nNumber of people with each disability type:")
print(df_clean[flag_cols].sum())

print("\nSample:")
df_clean[["disability_type"] + flag_cols].head(10)

Unique disability_type values:
disability_type
NaN                         692970
Vision difficulty            18981
Mental/other difficulty      13080
Walking difficulty           12749
Hearing difficulty            6130
Hand/finger difficulty        4714
Communication difficulty      2998
Name: count, dtype: int64

Disability type flags created:
['disability_vision', 'disability_mental', 'disability_walking', 'disability_hearing', 'disability_communication', 'disability_hand_finger', 'no_disability']

Number of people with each disability type:
disability_vision            18981
disability_mental            13080
disability_walking           12749
disability_hearing            6130
disability_communication      2998
disability_hand_finger        4714
no_disability               692970
dtype: int64

Sample:


,disability_type,disability_vision,disability_mental,disability_walking,disability_hearing,disability_communication,disability_hand_finger,no_disability
0,NaN,0,0,0,0,0,0,1
1,NaN,0,0,0,0,0,0,1
2,NaN,0,0,0,0,0,0,1
3,NaN,0,0,0,0,0,0,1
4,NaN,0,0,0,0,0,0,1
5,NaN,0,0,0,0,0,0,1
6,NaN,0,0,0,0,0,0,1
7,NaN,0,0,0,0,0,0,1
8,NaN,0,0,0,0,0,0,1
9,NaN,0,0,0,0,0,0,1


### 3.4 Digital Status (0 / 1 / 2)

In [12]:
def create_digital_status(row):
    """
    0 = Non-digital
    1 = Digital user but does not use internet
    2 = Uses internet
    """
    digital = str(row.get("digital_use", "")).lower()
    internet = str(row.get("internet_use", "")).lower()

    if "non-digital" in digital or "non digital" in digital or digital == "0":
        return 0
    # digital user
    if "do not use" in internet or "not use" in internet or internet in ["0", "no"]:
        return 1
    return 2

df_clean["digital_status"] = df_clean.apply(create_digital_status, axis=1)

print("digital_status distribution:")
print(df_clean["digital_status"].value_counts().sort_index())
print("\nCross-tab digital_use × internet_use (for verification):")
print(pd.crosstab(df_clean["digital_use"], df_clean["internet_use"], margins=True))

digital_status distribution:
digital_status
0    502103
1     43027
2    206492
Name: count, dtype: int64

Cross-tab digital_use × internet_use (for verification):
internet_use  Do not use internet  Use internet     All
digital_use                                            
Digital user                43027        206492  249519
All                         43027        206492  249519


### 3.5 Province → Region

In [13]:
# Standard BPS province codes → broad region
province_to_region = {
    # Sumatra
    11: "Sumatra", 12: "Sumatra", 13: "Sumatra", 14: "Sumatra", 15: "Sumatra",
    16: "Sumatra", 17: "Sumatra", 18: "Sumatra", 19: "Sumatra", 21: "Sumatra",
    # Java
    31: "Java", 32: "Java", 33: "Java", 34: "Java", 35: "Java", 36: "Java",
    # Bali & Nusa Tenggara
    51: "Bali_Nusa", 52: "Bali_Nusa", 53: "Bali_Nusa",
    # Kalimantan
    61: "Kalimantan", 62: "Kalimantan", 63: "Kalimantan", 64: "Kalimantan", 65: "Kalimantan",
    # Sulawesi
    71: "Sulawesi", 72: "Sulawesi", 73: "Sulawesi", 74: "Sulawesi", 75: "Sulawesi", 76: "Sulawesi",
    # Eastern Indonesia (Maluku + Papua)
    81: "Eastern", 82: "Eastern",
    91: "Eastern", 92: "Eastern", 94: "Eastern", 95: "Eastern", 96: "Eastern", 97: "Eastern",
}

df_clean["region"] = df_clean["province"].map(province_to_region)

print("Region distribution:")
print(df_clean["region"].value_counts(dropna=False))

unmapped = df_clean.loc[df_clean["region"].isna(), "province"].unique()
if len(unmapped) > 0:
    print("\nUnmapped province codes (please check):", unmapped)
else:
    print("\nAll provinces mapped successfully.")

Region distribution:
region
Sumatra       221393
Java          211872
Sulawesi      112059
Kalimantan     77717
Eastern        66748
Bali_Nusa      61833
Name: count, dtype: int64

All provinces mapped successfully.


### 3.6 Income Class (Target)

In [19]:
print("Before cleaning:")
print("Missing weekly_workhours :", df_clean["weekly_workhours"].isna().sum())
print("Missing monthly_income   :", df_clean["monthly_income"].isna().sum())

df_clean["workhours_missing"] = df_clean["weekly_workhours"].isna().astype(int)
df_clean["weekly_workhours"] = df_clean["weekly_workhours"].fillna(0)

# Income class based on BPS (Statistics Indonesia) 2024 household expenditure
# classes, per capita per month:
#   Middle class : Rp 2,040,262 - Rp 9,909,844
#   Upper class  : > Rp 9,909,844
# (source: BPS 2024, via databoks.katadata.co.id / Tempo 2024)
# No_income is specific to this labor-force dataset (BPS scale has no such tier).
MIDDLE_LOWER_BOUND = 2_040_262
UPPER_LOWER_BOUND = 9_909_844

df_clean["income_class"] = "No_income"

mask_valid = df_clean["monthly_income"].notna() & (df_clean["monthly_income"] > 0)

df_clean.loc[
    mask_valid & (df_clean["monthly_income"] < MIDDLE_LOWER_BOUND),
    "income_class"
] = "Lower"

df_clean.loc[
    mask_valid
    & (df_clean["monthly_income"] >= MIDDLE_LOWER_BOUND)
    & (df_clean["monthly_income"] < UPPER_LOWER_BOUND),
    "income_class"
] = "Middle"

df_clean.loc[
    mask_valid & (df_clean["monthly_income"] >= UPPER_LOWER_BOUND),
    "income_class"
] = "Upper"

print("Income class distribution:")
print(df_clean["income_class"].value_counts().sort_index())

print("\nworkhours_missing distribution:")
print(df_clean["workhours_missing"].value_counts())

print("\nIncome statistics by class (only for people with income):")
print(
    df_clean[df_clean["income_class"] != "No_income"]
    .groupby("income_class", observed=True)["monthly_income"]
    .describe()
    .round(0)
)

Before cleaning:
Missing weekly_workhours : 0
Missing monthly_income   : 325770
Income class distribution:
income_class
Lower        239425
Middle       175908
No_income    327963
Upper          8326
Name: count, dtype: int64

workhours_missing distribution:
workhours_missing
0    751622
Name: count, dtype: int64

Income statistics by class (only for people with income):
                 count        mean         std        min         25%         50%         75%          max
income_class                                                                                              
Lower         239425.0   1116378.0    563400.0        1.0    600000.0   1000000.0   1500000.0    2040000.0
Middle        175908.0   3764734.0   1484056.0  2041000.0   2700000.0   3200000.0   4500000.0    9900000.0
Upper           8326.0  17713279.0  27899368.0  9911000.0  10000000.0  12000000.0  17000000.0  900000000.0


In [20]:
near_boundary = df_clean[
    (df_clean["income_class"] == "Upper") &
    (df_clean["monthly_income"] < UPPER_LOWER_BOUND * 1.15)
]
print(f"Upper earners within 15% of the boundary: {len(near_boundary)} / {(df_clean['income_class']=='Upper').sum()}")

Upper earners within 15% of the boundary: 3800 / 8326


### 3.7 Drop Columns We No Longer Need

In [21]:
cols_to_drop = [
    # Original categoricals that we encoded
    "gender",
    "disability",
    "disability_severity",
    "disability_type",
    "digital_use",
    "internet_use",
    "social_participation",
    "agri_employment",
    "province",                    # replaced by region

    # Derived / redundant
    "agri_productivity",
    "agri_productivity_trimmed",
    "log_monthly_income",          # used only to understand distribution
    "monthly_workhours",           # keep weekly_workhours instead
    "monthly_income",              # target is now income_class

    # Temporary helper
    "is_digital_user",
]

# Only drop columns that actually exist
cols_to_drop = [c for c in cols_to_drop if c in df_clean.columns]
df_clean = df_clean.drop(columns=cols_to_drop)

print("Dropped columns:", cols_to_drop)
print("\nRemaining columns:")
print(df_clean.columns.tolist())
print("\nFinal shape:", df_clean.shape)

Dropped columns: ['gender', 'disability', 'disability_severity', 'disability_type', 'digital_use', 'internet_use', 'social_participation', 'agri_employment', 'province', 'agri_productivity', 'agri_productivity_trimmed', 'log_monthly_income', 'monthly_workhours', 'monthly_income', 'is_digital_user']

Remaining columns:
['age', 'education_years', 'weekly_workhours', 'digital_index', 'social_participation_index', 'gender_male', 'disabled', 'social_participant', 'agri_worker', 'disability_vision', 'disability_mental', 'disability_walking', 'disability_hearing', 'disability_communication', 'disability_hand_finger', 'no_disability', 'digital_status', 'region', 'workhours_missing', 'income_class']

Final shape: (751622, 20)


## 4. Final Checks

In [22]:
print("Missing values after cleaning:")
print(df_clean.isnull().sum().sort_values(ascending=False))

print("\n\nData types:")
print(df_clean.dtypes)

print("\n\nSample of cleaned data:")
df_clean.head()

Missing values after cleaning:
age                           0
education_years               0
weekly_workhours              0
digital_index                 0
social_participation_index    0
gender_male                   0
disabled                      0
social_participant            0
agri_worker                   0
disability_vision             0
disability_mental             0
disability_walking            0
disability_hearing            0
disability_communication      0
disability_hand_finger        0
no_disability                 0
digital_status                0
region                        0
workhours_missing             0
income_class                  0
dtype: int64


Data types:
age                           float64
education_years               float64
weekly_workhours              float64
digital_index                 float64
social_participation_index    float64
gender_male                     int64
disabled                        int64
social_participant              int6

,age,education_years,weekly_workhours,digital_index,social_participation_index,gender_male,disabled,social_participant,agri_worker,disability_vision,disability_mental,disability_walking,disability_hearing,disability_communication,disability_hand_finger,no_disability,digital_status,region,workhours_missing,income_class
0,78.0,6.0,0.0,0.0,0.333333,1,0,1,0,0,0,0,0,0,0,1,0,Sumatra,0,No_income
1,65.0,6.0,0.0,0.0,0.333333,0,0,1,0,0,0,0,0,0,0,1,0,Sumatra,0,No_income
2,82.0,6.0,0.0,0.0,0.000000,1,0,0,0,0,0,0,0,0,0,1,0,Sumatra,0,No_income
3,65.0,6.0,0.0,0.0,0.000000,0,0,0,0,0,0,0,0,0,0,1,0,Sumatra,0,No_income
4,31.0,12.0,35.0,1.0,0.333333,1,0,1,0,0,0,0,0,0,0,1,2,Sumatra,0,Lower


In [23]:
df_clean.columns.tolist()

['age',
 'education_years',
 'weekly_workhours',
 'digital_index',
 'social_participation_index',
 'gender_male',
 'disabled',
 'social_participant',
 'agri_worker',
 'disability_vision',
 'disability_mental',
 'disability_walking',
 'disability_hearing',
 'disability_communication',
 'disability_hand_finger',
 'no_disability',
 'digital_status',
 'region',
 'workhours_missing',
 'income_class']

In [26]:
# Does digital_index differ between near-boundary and far-boundary Upper earners?
upper_df = df_clean[df_clean['income_class'] == 'Upper'].copy()
upper_df['near_boundary'] = df['monthly_income'] < UPPER_LOWER_BOUND * 1.15

print(upper_df.groupby('near_boundary')['digital_index'].describe())

from scipy.stats import mannwhitneyu
near = upper_df[upper_df['near_boundary']]['digital_index']
far = upper_df[~upper_df['near_boundary']]['digital_index']
stat, p = mannwhitneyu(near, far)
print(f'Mann-Whitney p-value: {p:.4f}')

                count      mean       std  min  25%  50%  75%  max
near_boundary                                                     
False          4526.0  1.824569  0.971950  0.0  1.0  2.0  3.0  3.0
True           3800.0  1.763421  0.979699  0.0  1.0  2.0  3.0  3.0
Mann-Whitney p-value: 0.0048


In [27]:
from scipy.stats import mannwhitneyu
import numpy as np

upper_df = df_clean[df_clean['income_class'] == 'Upper'].copy()
upper_df['near_boundary'] = df['monthly_income'] < UPPER_LOWER_BOUND * 1.15

near = upper_df[upper_df['near_boundary']]
far = upper_df[~upper_df['near_boundary']]

# continuous / ordinal features worth testing this way
numeric_features = [
    'age', 'education_years', 'weekly_workhours', 'digital_index',
    'social_participation_index', 'digital_status'
]

# binary features, compare proportions instead of means, still via Mann-Whitney (works fine on 0/1)
binary_features = [
    'gender_male', 'disabled', 'social_participant', 'agri_worker',
    'disability_vision', 'disability_mental', 'disability_walking',
    'disability_hearing', 'disability_communication', 'disability_hand_finger',
    'no_disability', 'workhours_missing'
]

results = []
for col in numeric_features + binary_features:
    near_vals = near[col].dropna()
    far_vals = far[col].dropna()

    stat, p = mannwhitneyu(near_vals, far_vals)

    mean_near = near_vals.mean()
    mean_far = far_vals.mean()
    pooled_std = np.sqrt((near_vals.std()**2 + far_vals.std()**2) / 2)
    # Cliff's delta approximation via rank-biserial correlation from U statistic
    n1, n2 = len(near_vals), len(far_vals)
    rank_biserial = 1 - (2 * stat) / (n1 * n2)
    cohens_d = (mean_near - mean_far) / pooled_std if pooled_std > 0 else np.nan

    results.append({
        'feature': col,
        'mean_near_boundary': round(mean_near, 4),
        'mean_far_boundary': round(mean_far, 4),
        'diff': round(mean_near - mean_far, 4),
        'cohens_d': round(cohens_d, 4),
        'p_value': round(p, 5),
        'significant': p < 0.05
    })

results_df = pd.DataFrame(results).sort_values('cohens_d', key=abs, ascending=False)
print(results_df.to_string(index=False))

                   feature  mean_near_boundary  mean_far_boundary    diff  cohens_d  p_value  significant
                       age             44.6500            46.0968 -1.4468   -0.1336  0.00000         True
             digital_index              1.7634             1.8246 -0.0611   -0.0627  0.00476         True
            digital_status              1.7039             1.7340 -0.0300   -0.0465  0.03704         True
          weekly_workhours             45.6097            46.1432 -0.5334   -0.0304  0.26576        False
                  disabled              0.0361             0.0413 -0.0053   -0.0273  0.21606        False
             no_disability              0.9639             0.9587  0.0053    0.0273  0.21606        False
           education_years             13.0568            12.9463  0.1105    0.0271  0.82449        False
        disability_walking              0.0047             0.0064 -0.0017   -0.0224  0.31090        False
               gender_male              0.7955

In [28]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(upper_df['near_boundary'], upper_df['region'])
chi2, p_region, dof, expected = chi2_contingency(contingency)
print(f'\nRegion association with near_boundary: chi2={chi2:.2f}, p={p_region:.5f}')
print(contingency)


Region association with near_boundary: chi2=37.97, p=0.00000
region         Bali_Nusa  Eastern  Java  Kalimantan  Sulawesi  Sumatra
near_boundary                                                         
False                311      300  1757         604       526     1028
True                 246      272  1254         624       469      935


## 5. Save Cleaned Dataset

In [ ]:

df_clean.to_csv('datset_preprocessed_with_work_hour_missing.csv',index=False)

print(f"✅ Cleaned dataset saved to: current dir")
print(f"   Shape: {df_clean.shape}")
print(f"   Columns: {df_clean.columns.tolist()}")

✅ Cleaned dataset saved to: current dir
   Shape: (751622, 21)
   Columns: ['age', 'education_years', 'weekly_workhours', 'digital_index', 'social_participation_index', 'gender_male', 'disabled', 'social_participant', 'agri_worker', 'disability_severity_ord', 'disability_vision', 'disability_mental', 'disability_walking', 'disability_hearing', 'disability_communication', 'disability_hand_finger', 'no_disability', 'digital_status', 'region', 'workhours_missing', 'income_class']


## 6. Feature List for Modeling (Quick Reference)

**Target:** `income_class` (Low / Medium / High)

**Features (recommended):**
- `age`
- `education_years`
- `gender_male`
- `disabled`
- `disability_severity_ord`
- `disability_vision`, `disability_mental`, `disability_walking`, `disability_hearing`, `disability_hand_finger`
- `weekly_workhours`
- `digital_status` (0/1/2)
- `digital_index`
- `social_participant`
- `social_participation_index`
- `agri_worker`
- `region` (or one-hot versions)

